In [1]:
# مرحله 1: کلون کردن مخزن
!git clone https://github.com/mohammadnabia/DA_nnUNet.git

# مرحله 2: وارد فولدر پروژه
%cd DA_nnUNet

# مرحله 3: نصب پروژه در حالت editable
!pip install -e .


Cloning into 'DA_nnUNet'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (297/297), done.
remote: Compressing objects: 100% (270/270), done.
remote: Total 297 (delta 41), reused 267 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (297/297), 2.58 MiB | 20.79 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/DA_nnUNet
Obtaining file:///content/DA_nnUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!unzip -q "/content/drive/MyDrive/BraTS-peds2023/BraTS-PEDs-2023.zip" -d /content/BraTS_PEDs_2023


In [3]:
import os
import shutil
import json
import glob

source_dir = "/content/BraTS_PEDs_2023/ASNR-MICCAI-BraTS2023-PED-Challenge-TrainingData"
target_dir = "/content/imagesTs"
os.makedirs(target_dir, exist_ok=True)

with open('/content/drive/MyDrive/100_epoch_training_on_bratspeds_finetune/splits_final.json', 'r') as f:
    splits = json.load(f)

val_cases = splits[0]['val']  # Fold 0

for case in val_cases:
    case_path = os.path.join(source_dir, case)
    if not os.path.isdir(case_path):
        continue
    try:
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t1n.nii.gz"))[0], os.path.join(target_dir, f"{case}_0000.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t1c.nii.gz"))[0], os.path.join(target_dir, f"{case}_0001.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t2w.nii.gz"))[0], os.path.join(target_dir, f"{case}_0002.nii.gz"))
        shutil.copyfile(glob.glob(os.path.join(case_path, "*t2f.nii.gz"))[0], os.path.join(target_dir, f"{case}_0003.nii.gz"))
    except IndexError:
        print(f"⚠️ Missing modalities for case: {case}")

print("✅ Validation fold 0 cases copied to imagesTs.")


✅ Validation fold 0 cases copied to imagesTs.


In [4]:
!ls /content/drive/MyDrive/100_epoch_training_on_bratspeds_finetune


dataset_fingerprint.json  fold_0_epoch_30.pth  fold_0_epoch_80.pth
dataset.json		  fold_0_epoch_40.pth  fold_0_epoch_90.pth
fold_0_epoch_100.pth	  fold_0_epoch_50.pth  nnUNetPlans.json
fold_0_epoch_10.pth	  fold_0_epoch_60.pth  splits_final.json
fold_0_epoch_20.pth	  fold_0_epoch_70.pth


In [5]:
!mkdir -p /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/fold_0

# کپی checkpoint
!cp "/content/drive/MyDrive/100_epoch_training_on_bratspeds_finetune/fold_0_epoch_100.pth" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth

# کپی plans.json
!cp "/content/drive/MyDrive/100_epoch_training_on_bratspeds_finetune/nnUNetPlans.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/plans.json

# کپی dataset.json
!cp "/content/drive/MyDrive/100_epoch_training_on_bratspeds_finetune/dataset.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/

# کپی splits_final.json
!cp "/content/drive/MyDrive/100_epoch_training_on_bratspeds_finetune/splits_final.json" \
    /content/nnUNet_results_baseline/Dataset139_BraTS2023/nnUNetTrainer_TL_FTen_Custom_100epochs__nnUNetPlans__3d_fullres/


In [6]:
import os

os.environ['nnUNet_raw'] = "/content/nnUNet_raw_baseline"
os.environ['nnUNet_preprocessed'] = "/content/nnUNet_preprocessed_baseline"
os.environ['nnUNet_results'] = "/content/nnUNet_results_baseline"


In [7]:
!mkdir -p /content/nnUNet_raw_baseline
!mkdir -p /content/nnUNet_preprocessed_baseline
!mkdir -p /content/nnUNet_results_baseline


In [8]:
import re

file_path = '/content/DA_nnUNet/nnunetv2/inference/predict_from_raw_data.py'
with open(file_path, 'r') as f:
    code = f.read()

# تغییر خط prediction
code = re.sub(r'prediction, _ = self\.network\(x\)', 'prediction = self.network(x)', code)

with open(file_path, 'w') as f:
    f.write(code)

print("✅ خط network(x) اصلاح شد!")


✅ خط network(x) اصلاح شد!


قبل از احرای این بخش
تابع زیر رو حتما به TRAINER  اضافه کن


```

import os
import shutil
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

class nnUNetTrainer_TL_FTen_Custom_100epochs(nnUNetTrainer):
    """
    Trainer برای 100 epoch آموزش، ذخیره checkpoint هر 10 اپوک و فایل split.
    """
    def __init__(self, plans, configuration, fold, dataset_json, unpack_dataset=True, device='cuda'):
        super().__init__(plans, configuration, fold, dataset_json, unpack_dataset, device)
        self.num_epochs = 100
        self.params_to_train = ['encoder.', '.seg_layers.']

    def load_dataset(self):
        """
        Override to save split file in Google Drive after dataset is loaded.
        """
        super().load_dataset()
        split_src = os.path.join(self.dataset_dir, "splits_final.json")
        split_dst = "/content/drive/MyDrive/nnUNet_checkpoints/splits_final.json"
        if os.path.exists(split_src):
            shutil.copy(split_src, split_dst)
            self.print_to_log_file(f"✅ فایل split ذخیره شد در: {split_dst}", also_print_to_console=True)

    def initialize(self):
        """
        Override to freeze/unfreeze selected layers.
        """
        super().initialize()
        for name, param in self.network.named_parameters():
            param.requires_grad = any(i in name for i in self.params_to_train)

    def on_epoch_end(self):
        """
        ذخیره checkpoint هر 10 اپوک و در پایان آموزش.
        """
        super().on_epoch_end()
        epoch = self.current_epoch if hasattr(self, "current_epoch") else self.epoch
        if (epoch + 1) % 10 == 0 or (epoch + 1) == self.num_epochs:
            save_dir = "/content/drive/MyDrive/nnUNet_checkpoints"
            os.makedirs(save_dir, exist_ok=True)
            save_path = os.path.join(save_dir, f"fold_{self.fold}_epoch_{epoch + 1}.pth")
            print(f"🔔 ذخیره checkpoint در epoch {epoch + 1} ...")
            self.save_checkpoint(save_path)
            print(f"✅ مدل در {save_path} ذخیره شد.")

    def predict_sliding_window_return_logits(self, data):
        """
        Override برای حل مشکل ValueError: unpack
        """
        outputs = self.network(data)
        if isinstance(outputs, tuple):
            prediction = outputs[0]
        else:
            prediction = outputs
        return prediction
        
```

In [9]:
!nnUNetv2_predict \
  -i /content/imagesTs \
  -o /content/pred_fold0_noTTA \
  -d 139 \
  -c 3d_fullres \
  -f 0 \
  -tr nnUNetTrainer_TL_FTen_Custom_100epochs \
  --disable_tta



#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 20 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 20 cases that I would like to predict

Predicting BraTS-PED-00008-000:
perform_everything_on_device: True
100% 8/8 [00:02<00:00,  3.67it/s]
sending off prediction to background worker for resampling and export
done with BraTS-PED-00008-000

Predicting BraTS-PED-00021-000:
perform_everything_on_device: True
100% 8/8 [00:00<00:00, 18.67it/s]
sending off prediction to background worker for resampling and export
done with BraTS-PED-00021-000

Predicting BraTS-PED-00026-000:

این فرآیند روی 20 کیس فقط 1 دقیقه طول کشیدد

In [10]:
!ls /content/pred_fold0_noTTA


BraTS-PED-00008-000.nii.gz  BraTS-PED-00099-000.nii.gz
BraTS-PED-00021-000.nii.gz  BraTS-PED-00101-000.nii.gz
BraTS-PED-00026-000.nii.gz  BraTS-PED-00104-000.nii.gz
BraTS-PED-00042-000.nii.gz  BraTS-PED-00107-000.nii.gz
BraTS-PED-00050-000.nii.gz  BraTS-PED-00110-000.nii.gz
BraTS-PED-00055-000.nii.gz  BraTS-PED-00115-000.nii.gz
BraTS-PED-00063-000.nii.gz  BraTS-PED-00118-000.nii.gz
BraTS-PED-00078-000.nii.gz  BraTS-PED-00132-000.nii.gz
BraTS-PED-00079-000.nii.gz  dataset.json
BraTS-PED-00084-000.nii.gz  plans.json
BraTS-PED-00086-000.nii.gz  predict_from_raw_data_args.json
BraTS-PED-00096-000.nii.gz


In [11]:
!mkdir -p /content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT100epoch_80pruned_noTTA
!cp -r /content/pred_fold0_noTTA/* /content/drive/MyDrive/nnUNet_Pruned_Results_peds2023/BraTS-PEDS_FT100epoch_80pruned_noTTA/
